# obspec smoke test

`obspec` is a pure-protocol package: `typing.Protocol` interfaces for object
storage (the spec that `obstore` and friends implement). All examples are
offline — minimal in-memory clients structurally satisfying the protocols.

## Example 1: protocol surface inventory

In [ ]:
import obspec

expected = {
    "Get", "GetAsync", "GetRange", "GetRangeAsync", "GetRanges", "GetRangesAsync",
    "Head", "HeadAsync", "List", "ListAsync", "ListWithDelimiter", "ListWithDelimiterAsync",
    "Put", "PutAsync", "Delete", "DeleteAsync", "Copy", "CopyAsync", "Rename", "RenameAsync",
}
missing = expected - set(obspec.__all__)
assert not missing, f"missing protocols: {missing}"

# Each sync protocol exposes a snake_case method named after itself
for proto_name, method in [("Get", "get"), ("Put", "put"), ("List", "list"), ("Head", "head")]:
    assert callable(getattr(getattr(obspec, proto_name), method)), f"{proto_name}.{method} missing"

print(f"obspec {obspec.__version__}: {len(obspec.__all__)} exported names, all core protocols present")

## Example 2: structural typing — a minimal in-memory Put/List client

In [ ]:
# Example 2: a minimal in-memory client structurally satisfying Put + List
# Source: adapted from the List.list docstring example in obspec/_list.py (packaged wheel)
from datetime import datetime, timezone

import obspec

class MemClient:
    def __init__(self):
        self.objs: dict[str, bytes] = {}

    def put(self, path, file, **kwargs):
        self.objs[path] = bytes(file)
        return {"e_tag": None, "version": None}

    def list(self, prefix=None, *, offset=None):
        now = datetime.now(timezone.utc)
        yield [
            {"path": p, "last_modified": now, "size": len(b), "e_tag": None, "version": None}
            for p, b in sorted(self.objs.items())
        ]

def upload_files(client: obspec.Put):
    for i in range(3):
        client.put(f"file{i}.txt", b"foo")

def list_files(client: obspec.List):
    return [meta["path"] for chunk in client.list() for meta in chunk]

mem = MemClient()
upload_files(mem)
assert list_files(mem) == ["file0.txt", "file1.txt", "file2.txt"]
print("structural Put/List round-trip:", list_files(mem))


## Example 3: typed request/response shapes

In [ ]:
from datetime import datetime, timezone

import obspec

# All GetOptions keys are optional; range accepts (start, end) / offset / suffix forms
assert obspec.GetOptions.__required_keys__ == frozenset()
assert "range" in obspec.GetOptions.__optional_keys__
opts: obspec.GetOptions = {"if_none_match": "*", "range": {"offset": 1024}}
suffix_opts: obspec.GetOptions = {"range": obspec.SuffixRange(suffix=256)}

# ObjectMeta requires the full metadata shape
assert obspec.ObjectMeta.__required_keys__ == {"path", "last_modified", "size", "e_tag", "version"}
meta = obspec.ObjectMeta(
    path="a/b.bin", last_modified=datetime.now(timezone.utc), size=3, e_tag="x", version=None
)

result: obspec.PutResult = {"e_tag": "abc", "version": "1"}
print("GetOptions:", opts, suffix_opts)
print("ObjectMeta keys:", sorted(meta))
print("PutResult:", result)

## Example 4: well-known exceptions + `map_exception`

In [ ]:
# Example 4: well-known exception names + map_exception
# Source: adapted from the module docstring of obspec/exceptions.py (packaged wheel)
from obspec import Head
from obspec.exceptions import BaseError, NotFoundError, map_exception

class MissingClient:
    def head(self, path):
        raise FileNotFoundError(f"no such object: {path}")

def check_if_exists(client: Head, path: str) -> bool:
    try:
        client.head(path)
    except Exception as e:
        if isinstance(map_exception(e), NotFoundError):
            return False
        raise
    return True

assert check_if_exists(MissingClient(), "gone.txt") is False
# NotFoundError doubles as the builtin FileNotFoundError; unknown errors pass through
assert issubclass(NotFoundError, FileNotFoundError) and issubclass(NotFoundError, BaseError)
mapped = map_exception(KeyError("boom"))
assert isinstance(mapped, KeyError) and not isinstance(mapped, BaseError)
print("map_exception:", type(map_exception(FileNotFoundError("x"))).__name__)


## Example 5: async protocol variants (top-level await)

In [ ]:
import obspec

class AsyncMemClient:
    def __init__(self):
        self.objs: dict[str, bytes] = {}

    async def put_async(self, path, file, **kwargs):
        self.objs[path] = bytes(file)
        return {"e_tag": None, "version": None}

    async def delete_async(self, paths):
        for p in [paths] if isinstance(paths, str) else paths:
            del self.objs[p]

async def mirror(client: obspec.PutAsync, payloads: dict[str, bytes]):
    for path, data in payloads.items():
        await client.put_async(path, data)

client = AsyncMemClient()
await mirror(client, {"a.bin": b"\x00\x01", "b.bin": b"\x02"})
assert set(client.objs) == {"a.bin", "b.bin"}
await client.delete_async("a.bin")
assert set(client.objs) == {"b.bin"}
print("async put/delete round-trip OK:", client.objs)